# Project 2 - Goodreads Recommendation System

**Group:** Vincent Pape, Aditya Bagchi and Jorge Zapata  
**Course:** AI Modeling in Practice  
**Professor:** Zafari

Goodreads is a social cataloging platform where users rate and review books.
Our objective is to explore the Goodreads dataset, build collaborative-filtering
recommendation models, evaluate them against a baseline, and prepare the
recommendation logic for a Streamlit app with an AI personalization layer.

We used the course Week 3 recommender-system notebooks as the main structure:
load the long-format ratings, build a Surprise dataset, compare a baseline with
user-based and item-based collaborative filtering, and evaluate both rating
prediction and Top-N ranking quality.

Note: We used AI assistance to help scaffold and debug the code. We adapted the
implementation and completed our own analysis and interpretation.

---
# Q1: Explore the dataset and provide insights about users, ratings, and books.



### Step 0: Setup



In [ ]:
# Upload the CSVs in Colab (Books.csv and Ratings.csv for Project 2)
from google.colab import files
uploaded = files.upload()


In [ ]:
# Install the recommender-system package used for Baseline, User-Based CF, and Item-Based CF.
# Run this cell before importing from surprise in Google Colab.
!pip install scikit-surprise


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from surprise import KNNBasic, BaselineOnly, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split

# Fix the seed so the train/test split and model comparison are reproducible.
RANDOM_STATE = 6604



### Load the Data



In [ ]:
# The assignment says the project should reproduce by reading Books.csv and Ratings.csv.
books = pd.read_csv("Books.csv")
ratings = pd.read_csv("Ratings.csv")

print("Books:", books.shape)
print("Ratings:", ratings.shape)
display(books.head())
display(ratings.head())



### Dataset Overview



In [ ]:
print("Books columns:")
print(books.columns.tolist())

print("\nRatings columns:")
print(ratings.columns.tolist())

print("\nRatings summary:")
display(ratings["rating"].describe())



In [ ]:
# Check missing values in the fields most important for modeling and reporting.
book_key_cols = ["book_id", "title", "authors", "average_rating", "ratings_count"]
rating_key_cols = ["book_id", "user_id", "rating"]

print("Missing values in book fields:")
display(books[book_key_cols].isna().sum().sort_values(ascending=False))

print("Missing values in rating fields:")
display(ratings[rating_key_cols].isna().sum().sort_values(ascending=False))



In [ ]:
# Count how many ratings each user and each book has.
# This matters because collaborative filtering works better when users/items
# have enough rating history to compare.
ratings_per_user = ratings.groupby("user_id")["rating"].count().sort_values(ascending=False)
ratings_per_book = ratings.groupby("book_id")["rating"].count().sort_values(ascending=False)

eda_summary = pd.DataFrame({
    "Metric": [
        "Total ratings",
        "Unique users",
        "Books in catalog",
        "Books with at least one rating",
        "Average ratings per user",
        "Median ratings per user",
        "Average ratings per book",
        "Median ratings per book",
    ],
    "Value": [
        len(ratings),
        ratings["user_id"].nunique(),
        len(books),
        ratings["book_id"].nunique(),
        ratings_per_user.mean(),
        ratings_per_user.median(),
        ratings_per_book.mean(),
        ratings_per_book.median(),
    ],
})
display(eda_summary)



In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(ratings["rating"], bins=np.arange(1, 7) - 0.5, edgecolor="black")
plt.xticks([1, 2, 3, 4, 5])
plt.xlabel("Rating")
plt.ylabel("Count")
plt.title("Distribution of Goodreads Ratings")
plt.show()



In [ ]:
# Most-rated books in the project sample.
top_books = (
    ratings_per_book.rename("project_rating_count")
    .reset_index()
    .merge(
        books[["book_id", "title", "authors", "average_rating", "ratings_count"]],
        on="book_id",
        how="left",
    )
    .sort_values("project_rating_count", ascending=False)
    .head(15)
)
display(top_books)



In [ ]:
# Highly rated books with a large overall Goodreads rating count.
# This gives a simple popularity/quality view before modeling.
books["ratings_count"] = pd.to_numeric(books["ratings_count"], errors="coerce")
books["average_rating"] = pd.to_numeric(books["average_rating"], errors="coerce")

popular_high_rating = (
    books[books["ratings_count"] >= 100000]
    .sort_values(["average_rating", "ratings_count"], ascending=False)
    [["book_id", "title", "authors", "average_rating", "ratings_count"]]
    .head(15)
)
display(popular_high_rating)



### Brief Insight

The Goodreads data is a classic recommendation problem because it has many
users, many books, and a long-format table of user-book ratings. The EDA step is
important because collaborative filtering depends on overlap: if users or books
have very few ratings, the model has less evidence for finding similar users or
similar items. The rating distribution also shows whether a simple popularity or
average-rating baseline may be hard to beat.

---
# Q2: Compare and evaluate recommendation models.



### Build the Surprise Dataset and Train/Test Split



In [ ]:
# Goodreads ratings use a 1-5 scale.
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[["user_id", "book_id", "rating"]], reader)

# Hold out 10% of the ratings so the models are evaluated on ratings they did
# not see during training.
trainset, testset = train_test_split(data, test_size=0.1, random_state=RANDOM_STATE)
print(f"Train: {trainset.n_ratings:,} ratings")
print(f"Test: {len(testset):,} ratings")



### Precision and Recall @ K Helper

A book is counted as relevant if the user's true held-out rating is at least
4.0. Precision@10 measures how many of the model's top 10 predictions were
relevant. Recall@10 measures how many of the user's relevant held-out books were
captured in that top 10 list.



In [ ]:
def precision_recall_at_k(predictions, top_n=10, threshold=4.0):
    user_data = defaultdict(list)

    # Group each prediction by user so we can sort that user's recommendations.
    for uid, _, true_r, est, _ in predictions:
        user_data[uid].append((est, true_r))

    precisions, recalls = [], []
    for items in user_data.values():
        items.sort(key=lambda x: x[0], reverse=True)

        n_relevant = sum(1 for _, true_r in items if true_r >= threshold)
        n_hits = sum(1 for _, true_r in items[:top_n] if true_r >= threshold)

        precisions.append(n_hits / top_n)
        if n_relevant > 0:
            recalls.append(n_hits / n_relevant)

    return np.mean(precisions), np.mean(recalls)



### Define Candidate Models



In [ ]:
# BaselineOnly is the popularity/mean benchmark from Surprise.
# KNNBasic is used for both user-based and item-based collaborative filtering.
models = {
    "Baseline": BaselineOnly(verbose=False),
    "UBCF cosine k=10": KNNBasic(
        k=10,
        sim_options={"name": "cosine", "user_based": True},
        verbose=False,
    ),
    "UBCF pearson k=50": KNNBasic(
        k=50,
        sim_options={"name": "pearson", "user_based": True},
        verbose=False,
    ),
    "IBCF cosine k=50": KNNBasic(
        k=50,
        sim_options={"name": "cosine", "user_based": False},
        verbose=False,
    ),
}



### Model Performance Evaluation



In [ ]:
results = []
fitted_models = {}

for name, model in models.items():
    print("=" * 50)
    print(name)
    print("=" * 50)

    model.fit(trainset)
    predictions = model.test(testset)

    rmse = accuracy.rmse(predictions, verbose=False)
    precision, recall = precision_recall_at_k(predictions, top_n=10, threshold=4.0)

    results.append({
        "Model": name,
        "RMSE": rmse,
        "Precision@10": precision,
        "Recall@10": recall,
    })
    fitted_models[name] = model

comparison = (
    pd.DataFrame(results)
    .sort_values(["Precision@10", "Recall@10", "RMSE"], ascending=[False, False, True])
    .reset_index(drop=True)
)
display(comparison.round(4))



In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(comparison["Model"][::-1], comparison["Precision@10"][::-1])
plt.xlabel("Precision@10")
plt.title("Top-N Ranking Performance by Model")
plt.show()

best_model_name = comparison.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
print("Selected model for recommendation list:", best_model_name)



### Brief Insight

RMSE evaluates rating prediction accuracy, while Precision@10 and Recall@10
evaluate the actual Top-N recommendation experience. For this assignment, we
should discuss both. If the baseline has the best RMSE, that means the average
rating structure is strong. If a CF model has stronger Precision@10 or Recall@10,
that model may still be better for the app because the product is showing a
ranked list of books, not just predicting a numeric rating.

---
# Q3: Generate Top-N recommendations from the selected collaborative-filtering model.



### Recommendation Helper



In [ ]:
book_lookup = books.set_index("book_id").to_dict(orient="index")
book_rating_counts = ratings["book_id"].value_counts()
book_mean_ratings = ratings.groupby("book_id")["rating"].mean()


def recommend_books(model, user_id, min_ratings=20, top_n=10):
    """Recommend books the user has not already rated."""
    popular_books = set(book_rating_counts[book_rating_counts >= min_ratings].index)
    seen_books = set(ratings.loc[ratings["user_id"] == user_id, "book_id"])

    candidates = []
    for book_id in books["book_id"]:
        if book_id in seen_books or book_id not in popular_books:
            continue

        meta = book_lookup.get(book_id, {})
        candidates.append({
            "book_id": book_id,
            "title": meta.get("title", ""),
            "authors": meta.get("authors", ""),
            "predicted_rating": model.predict(user_id, book_id).est,
            "project_rating_count": int(book_rating_counts.get(book_id, 0)),
            "project_average_rating": float(book_mean_ratings.get(book_id, np.nan)),
            "goodreads_average_rating": meta.get("average_rating", np.nan),
        })

    candidates.sort(key=lambda row: row["predicted_rating"], reverse=True)
    return pd.DataFrame(candidates[:top_n])



In [ ]:
# Pick a sample user with many ratings so the demo has enough history to learn from.
sample_user = int(ratings_per_user.index[0])
print("Sample user:", sample_user)

sample_recommendations = recommend_books(best_model, sample_user, min_ratings=20, top_n=10)
display(sample_recommendations)



In [ ]:
sample_history = (
    ratings[ratings["user_id"] == sample_user]
    .merge(books, on="book_id", how="left")
    [["title", "authors", "rating"]]
    .sort_values("rating", ascending=False)
    .reset_index(drop=True)
)
display(sample_history.head(20))



### Brief Insight

The recommendation helper excludes books the user has already rated, applies a
minimum-ratings filter to avoid unstable recommendations, and sorts unseen
books by the selected model's predicted rating. This is the same candidate
generation logic that the Streamlit app uses.

---
# Q4: Add an AI re-ranking layer on top of the recommender.



### LLM Personalization Strategy

The LLM should not create a new recommendation list from scratch. Instead, the
collaborative-filtering model first retrieves a Top-N candidate set, and the LLM
re-ranks only those candidate books using book metadata such as title, author,
publication year, average rating, and predicted rating.

Example prompt strategy used in the app:

1. Give the LLM the user's stated preference, such as mood, genre, or reading
   goal.
2. Provide only the collaborative-filtering candidate books.
3. Instruct the LLM not to add new books.
4. Ask for structured JSON output with a short explanation for each pick.

This design keeps the CF model responsible for recommendation retrieval and
uses the LLM for personalization and explanation.



### Brief Insight

The AI layer improves the user experience by translating a generic Top-N list
into a preference-aware list. From a business perspective, this makes the app
feel more personalized and explainable. The main risks are prompt reliability,
API cost/latency, and preventing the LLM from inventing recommendations outside
the candidate list.

---
# Q5: Build a Streamlit app and video demo.



### App Structure

The file `app.py` follows the Week 3 Streamlit recommender demo:

- `load_data()` reads `Books.csv` and `Ratings.csv`.
- `fit_cf_model()` fits a baseline, user-based CF, or item-based CF model.
- `recommend()` creates the collaborative-filtering Top-N candidate list.
- `rerank_with_gemini()` optionally re-ranks those candidates with Gemini.
- The sidebar lets the user choose a user ID, model, minimum rating count, and
  list size.
- The main page shows CF recommendations, optional AI re-ranking, and the
  selected user's rating history.

To run locally:

```bash
pip install -r requirements.txt
streamlit run app.py
```

Do not submit an API key. For Gemini, use an environment variable named
`GEMINI_API_KEY` or a local `.streamlit/secrets.toml` file that is not committed.



### Brief Insight

The app is designed for the required video demo. In the walkthrough, we can show
one user, explain the CF candidate list, enter a stated reading preference, and
show how Gemini re-ranks the same books with short explanations.

---
# Q6: Business discussion, conclusion, and submitted items.



### Business Interpretation

Collaborative filtering is useful for platforms such as Goodreads because it
learns from reader behavior instead of relying only on manually labeled genres.
User-based CF is easy to explain as "readers similar to you liked these books."
Item-based CF is useful when a business wants stable item-to-item relationships,
such as "people who liked this book also liked that book."

The Week 2 responsible AI material also applies here. A recommender should be
monitored for sparse-data problems, popularity bias, and cold-start users or
books that get weak recommendations because they have limited history.
Explanations should be truthful about what the model used: collaborative
filtering uses rating patterns, while the LLM layer uses only the candidate
metadata provided to it.

The LLM personalization layer has a different business role. It can adjust a
recommendation list to a user's current mood or stated preference and provide
natural-language explanations. This can make recommendations feel more useful,
but it also introduces operational challenges: API key management, cost,
latency, output consistency, and the need to prevent hallucinated items.

### Conclusion

The recommended approach is to use the best collaborative-filtering model from
the evaluation table as the retrieval model, then use the LLM only as a
re-ranking and explanation layer. This follows the assignment requirement and
keeps the system grounded in actual Goodreads candidates.

### Submitted Items

- Final code notebook / Python code
- Streamlit app code
- PDF slide deck, maximum 8 slides plus optional appendix slides
- Shareable 2-3 minute video demo link

### AI Usage Citation

AI assistance was used for coding support, debugging, and revising
implementation details. Final modeling decisions, interpretation, and submitted
analysis should be reviewed and completed by the project team.
